In [11]:
import numpy as np
import pandas as pd
import models
from database import engine, sessionDB
from nba_api.stats.static import players
from nba_api.stats.endpoints import playerawards

In [12]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [13]:
players= players.get_players()
#players

In [14]:
models.Base.metadata.create_all(bind=engine)

award testing and structure allat stuff

In [ ]:
#using lebron cus hes the goat
lebron = [p for p in players if p["full_name"].lower() == "lebron james"]
lebron = lebron[0] #de-listifying (dont do this in prod)
lebron_id = lebron["id"] #2544
lebron_awards = playerawards.PlayerAwards(player_id=lebron_id).get_data_frames()[0]

acronym_hmap = {
    "All-Defensive Team": "DEF",''
    "All-NBA": "NBA",
    "All-Rookie Team": "ROOK",
    "NBA All-Star": "AS",
    "NBA All-Star Most Valuable Player": "ASMVP",
    "NBA Champion": "CHAMP",
    "NBA Cup All-Tournament Team": "CUP",
    "NBA Cup Most Valuable Player": "CUPMVP",
    "NBA Finals Most Valuable Player": "FMVP",
    "NBA Most Valuable Player": "MVP",
    "NBA Player of the Month": "POTM",
    "NBA Player of the Week": "POTW",
    "NBA Rookie of the Month": "ROTM",
    "NBA Rookie of the Year": "ROTY",
}

all_nba_set = {"DEF", "NBA", "ROOK", "CUP"}

#gotta add stat champs + dpoy/dpom + cfmvp + cpoy + dunk/3pt contest + teammate oty + hustle player oty

clean_lebron_awards_df = lebron_awards[lebron_awards["DESCRIPTION"].isin(acronym_hmap)]
clean_lebron_awards_df = clean_lebron_awards_df.drop([
    "FIRST_NAME", 
    "LAST_NAME", 
    "CONFERENCE", 
    "TYPE", 
    "SUBTYPE1", 
    "SUBTYPE2", 
    "SUBTYPE3",
    "MONTH",
    "WEEK",
    "PERSON_ID",
    "TEAM"
], axis=1)

clean_lebron_awards_df["ALL_NBA_TEAM_NUMBER"] = clean_lebron_awards_df["ALL_NBA_TEAM_NUMBER"].fillna(0)
clean_lebron_awards_df = clean_lebron_awards_df.replace(r'^\s*$', 0, regex=True) 
#^ gets rid of empty strings

clean_lebron_awards_df["DESCRIPTION"] = clean_lebron_awards_df["DESCRIPTION"].map(acronym_hmap)
clean_lebron_awards_df["DESCRIPTION"] = clean_lebron_awards_df.apply(
    lambda r: f"{r['DESCRIPTION']}{r['ALL_NBA_TEAM_NUMBER']}" 
    if r['DESCRIPTION'] in all_nba_set else r['DESCRIPTION'], 
    axis=1
)

clean_lebron_awards_df

,DESCRIPTION,ALL_NBA_TEAM_NUMBER,SEASON
0,DEF1,1,2008-09
1,DEF1,1,2009-10
2,DEF1,1,2010-11
3,DEF1,1,2011-12
4,DEF1,1,2012-13
5,DEF2,2,2013-14
6,NBA2,2,2004-05
7,NBA1,1,2005-06
8,NBA2,2,2006-07
9,NBA1,1,2007-08


actual award parsing

In [ ]:
def getAwards(pid: int):
    awards_df = playerawards.PlayerAwards(player_id=pid).get_data_frames()[0]

    acronym_hmap = {
        "All-Defensive Team": "DEF",''
        "All-NBA": "NBA",
        "All-Rookie Team": "ROOK",
        "NBA All-Star": "AS",
        "NBA All-Star Most Valuable Player": "ASMVP",
        "NBA Champion": "CHAMP",
        "NBA Cup All-Tournament Team": "CUP",
        "NBA Cup Most Valuable Player": "CUPMVP",
        "NBA Finals Most Valuable Player": "FMVP",
        "NBA Most Valuable Player": "MVP",
        "NBA Player of the Month": "POTM",
        "NBA Player of the Week": "POTW",
        "NBA Rookie of the Month": "ROTM",
        "NBA Rookie of the Year": "ROTY",
    }

    all_nba_set = {"DEF", "NBA", "ROOK", "CUP"}

    awards_df = awards_df[awards_df["DESCRIPTION"].isin(acronym_hmap)]
    awards_df = awards_df.drop([
        "FIRST_NAME", 
        "LAST_NAME", 
        "CONFERENCE", 
        "TYPE", 
        "SUBTYPE1", 
        "SUBTYPE2", 
        "SUBTYPE3",
        "MONTH",
        "WEEK",
        "PERSON_ID",
        "TEAM"
    ], axis=1)

    awards_df["ALL_NBA_TEAM_NUMBER"] = awards_df["ALL_NBA_TEAM_NUMBER"].fillna(0)
    awards_df = awards_df.replace(r'^\s*$', 0, regex=True) 
    #^ gets rid of empty strings

    awards_df["DESCRIPTION"] = awards_df["DESCRIPTION"].map(acronym_hmap)
    awards_df["DESCRIPTION"] = awards_df.apply(
        lambda r: f"{r['DESCRIPTION']}{r['ALL_NBA_TEAM_NUMBER']}" 
        if r['DESCRIPTION'] in all_nba_set else r['DESCRIPTION'], 
        axis=1
    )

    return awards_df.drop(["ALL_NBA_TEAM_NUMBER"], axis=1)

getAwards(2544)




,DESCRIPTION,SEASON
0,DEF1,2008-09
1,DEF1,2009-10
2,DEF1,2010-11
3,DEF1,2011-12
4,DEF1,2012-13
5,DEF2,2013-14
6,NBA2,2004-05
7,NBA1,2005-06
8,NBA2,2006-07
9,NBA1,2007-08
